# IMDb Screenplay Success Classifier — Fine-tuned Transformer

This notebook switches the task from **exact IMDb rating regression** to **success classification**.

The previous regression model collapsed toward the mean because exact IMDb rating is noisy from screenplay text alone. This model instead predicts:

```python
success = rating >= RATING_THRESHOLD
```

Default threshold: `7.0`.

It fine-tunes a transformer on screenplay chunks, aggregates chunk probabilities back to the movie level, and reports classification metrics.


In [1]:
# ============================================================
# 1) INSTALL COMPATIBLE PACKAGES FOR BASE PYTHON 3.12 / ANACONDA
# ============================================================
# Run this cell first. If it installs/updates packages, restart the kernel once,
# then run all cells again.
#
# IMPORTANT:
# - No sentence-transformers dependency here.
# - We use HuggingFace transformers + PyTorch directly.
# - TensorFlow/Keras is disabled because Keras 3 can break transformers imports.

import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "scipy": "1.13.1",
    "scikit-learn": "1.6.1",
    "joblib": "1.4.2",
    "tqdm": "4.67.1",
    "torch": "2.5.1",
    "transformers": "4.46.3",
    "huggingface-hub": "0.26.5",
    "tokenizers": "0.20.3",
    "safetensors": "0.4.5",
}

def installed_version(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

to_install = []
for pkg, wanted_version in REQUIRED.items():
    current_version = installed_version(pkg)
    if current_version != wanted_version:
        to_install.append(f"{pkg}=={wanted_version}")

if to_install:
    print("Installing/fixing packages:")
    for item in to_install:
        print("  ", item)

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        *to_install,
    ])

    print("\nInstalled/fixed packages.")
    print("IMPORTANT: restart the kernel now, then Run All again.")
else:
    print("Package versions already look compatible. Continue to imports.")


Package versions already look compatible. Continue to imports.


In [2]:
# ============================================================
# 2) IMPORTS
# ============================================================

import os

# Disable TensorFlow/Keras paths inside transformers.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

from pathlib import Path
import re
import math
import random
import json
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import joblib

from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

print("Imports OK")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)


Imports OK
numpy: 1.26.4
pandas: 2.2.3
torch: 2.5.1+cpu
sklearn: 1.6.1


In [3]:
# ============================================================
# 3) CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Main target. Change to 7.5 for stricter "high-success" classification.
RATING_THRESHOLD = 7.0

# Transformer model. DistilBERT is a reasonable laptop-friendly BERT-style model.
MODEL_NAME = "distilbert-base-uncased"

# Training controls. Increase EPOCHS to 4-5 if it is stable and not overfitting.
MAX_MOVIE_CHUNKS = 32
WORDS_PER_CHUNK = 220
MAX_TOKEN_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.25
UNFREEZE_LAST_N_LAYERS = 2

# Output files
PREDICTIONS_OUT_PATH = Path("bert_success_classifier_movie_predictions.csv")
CHUNK_PREDICTIONS_OUT_PATH = Path("bert_success_classifier_chunk_predictions.csv")
MODEL_OUT_PATH = Path("bert_success_classifier_model.pt")

print("Rating threshold:", RATING_THRESHOLD)
print("Model:", MODEL_NAME)


Device: cpu
Rating threshold: 7.0
Model: distilbert-base-uncased


In [4]:
# ============================================================
# 4) PATH SETUP
# ============================================================

def find_repo_root():
    current = Path.cwd().resolve()

    candidates = [
        current,
        current.parent,
        current.parent.parent,
    ]

    for candidate in candidates:
        if (candidate / "movie_ratings_final.csv").exists() and (candidate / "screenplay_dataset").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find repo root. Put this notebook in the repo root or notebooks/ folder.\n"
        "Expected files:\n"
        "  movie_ratings_final.csv\n"
        "  screenplay_dataset/"
    )

REPO_DIR = find_repo_root()
CSV_PATH = REPO_DIR / "movie_ratings_final.csv"
SCREENPLAY_DIR = REPO_DIR / "screenplay_dataset"

print("Repo dir:", REPO_DIR)
print("CSV:", CSV_PATH)
print("Screenplay dir:", SCREENPLAY_DIR)


Repo dir: C:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure
CSV: C:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure\movie_ratings_final.csv
Screenplay dir: C:\Users\RiceC\IMDB\Predicting-Screenplay-Success-Using-Film-Structure\screenplay_dataset


In [5]:
# ============================================================
# 5) LOAD RATINGS + LOCAL SCREENPLAY TEXT
# ============================================================

def normalize_title(title):
    title = str(title).lower().strip()
    title = title.replace("&", " and ")
    title = re.sub(r"\([^)]*\)", " ", title)
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"^(the|a|an)\s+", "", title)
    return title


def find_col(df, possible_names):
    lower_map = {col.lower().strip(): col for col in df.columns}

    for name in possible_names:
        key = name.lower().strip()
        if key in lower_map:
            return lower_map[key]

    return None


def clean_screenplay_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def build_screenplay_index(screenplay_dir):
    file_map = {}

    txt_files = list(screenplay_dir.rglob("*.txt"))
    print(f"Found {len(txt_files):,} .txt files under screenplay_dataset/")

    for path in txt_files:
        stem = path.stem

        # Normalize common repo filename style:
        # Script_Matrix, The.txt -> matrix
        stem = re.sub(r"^Script_", "", stem)
        stem = re.sub(r",\s*(The|A|An|L'|El|La|Le)$", "", stem)

        norm = normalize_title(stem)

        if norm and norm not in file_map:
            file_map[norm] = path

    return file_map


def load_script_text(title, file_map):
    norm = normalize_title(title)

    if norm in file_map:
        return file_map[norm].read_text(encoding="utf-8", errors="ignore")

    # Backup direct filename guesses.
    title = str(title)
    direct_candidates = [
        SCREENPLAY_DIR / f"Script_{title}.txt",
        SCREENPLAY_DIR / f"{title}.txt",
    ]

    for path in direct_candidates:
        if path.exists():
            return path.read_text(encoding="utf-8", errors="ignore")

    return ""


ratings_df = pd.read_csv(CSV_PATH)
print("CSV columns:", list(ratings_df.columns))

title_col = find_col(ratings_df, ["original_title", "title", "movie_title", "name"])
rating_col = find_col(ratings_df, ["rating", "imdb_rating", "IMDb Rating", "averageRating"])
text_col = find_col(ratings_df, ["script_text", "screenplay_text", "cleaned_text", "text", "screenplay", "raw_text"])

if title_col is None:
    raise ValueError(f"Could not find title column. Columns: {list(ratings_df.columns)}")

if rating_col is None:
    raise ValueError(f"Could not find rating column. Columns: {list(ratings_df.columns)}")

ratings_df[rating_col] = pd.to_numeric(ratings_df[rating_col], errors="coerce")
ratings_df = ratings_df.dropna(subset=[title_col, rating_col]).reset_index(drop=True)

if text_col is not None and ratings_df[text_col].notna().sum() > 0:
    print(f"Using screenplay text from CSV column: {text_col}")
    ratings_df["screenplay_text"] = ratings_df[text_col].apply(clean_screenplay_text)
else:
    print("No screenplay text column found in CSV. Reading local .txt scripts.")
    screenplay_index = build_screenplay_index(SCREENPLAY_DIR)
    ratings_df["screenplay_text"] = [
        clean_screenplay_text(load_script_text(title, screenplay_index))
        for title in tqdm(ratings_df[title_col], desc="Reading local scripts")
    ]

ratings_df["word_count"] = ratings_df["screenplay_text"].str.split().str.len()
movie_df = ratings_df[ratings_df["word_count"] >= 500].copy().reset_index(drop=True)

movie_df["movie_id"] = np.arange(len(movie_df))
movie_df["label"] = (movie_df[rating_col] >= RATING_THRESHOLD).astype(int)
movie_df["norm_title"] = movie_df[title_col].apply(normalize_title)

print("Usable movies:", len(movie_df))
print("Rating mean:", movie_df[rating_col].mean())
print("Rating std:", movie_df[rating_col].std())
print("Class counts:")
display(movie_df["label"].value_counts().rename(index={0: f"< {RATING_THRESHOLD}", 1: f">= {RATING_THRESHOLD}"}).to_frame("count"))

if len(movie_df) < 50:
    raise ValueError("Too few usable screenplay texts. Check screenplay_dataset/ matching.")


CSV columns: ['original_title', 'rating', 'is_successful']
No screenplay text column found in CSV. Reading local .txt scripts.
Found 1,092 .txt files under screenplay_dataset/


Reading local scripts:   0%|          | 0/1076 [00:00<?, ?it/s]

Usable movies: 1036
Rating mean: 6.9101351351351346
Rating std: 0.9309511945576874
Class counts:


,count
label,
>= 7.0,548
< 7.0,488


In [6]:
# ============================================================
# 6) SHOW CLASS BALANCE FOR POSSIBLE THRESHOLDS
# ============================================================

threshold_rows = []

for threshold in [6.5, 7.0, 7.25, 7.5, 8.0]:
    labels = (movie_df[rating_col] >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "negative_count": int((labels == 0).sum()),
        "positive_count": int((labels == 1).sum()),
        "positive_rate": float(labels.mean()),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)


,threshold,negative_count,positive_count,positive_rate
0,6.50,290,746,0.720077
1,7.00,488,548,0.528958
2,7.25,635,401,0.387066
3,7.50,733,303,0.292471
4,8.00,922,114,0.110039


In [7]:
# ============================================================
# 7) CHUNK SCREENPLAYS INTO TRAINING EXAMPLES
# ============================================================

def screenplay_to_chunks(text, max_chunks=32, words_per_chunk=220):
    # Split screenplay into ordered word windows. We sample across the whole script
    # so beginning/middle/end are all represented.
    # Each chunk inherits the movie's label.
    text = clean_screenplay_text(text)
    words = text.split()

    if len(words) == 0:
        return []

    if len(words) <= words_per_chunk:
        return [" ".join(words)]

    max_start = max(0, len(words) - words_per_chunk)
    starts = np.linspace(0, max_start, max_chunks).astype(int)

    chunks = []
    seen = set()

    for start in starts:
        chunk_words = words[start:start + words_per_chunk]
        chunk = " ".join(chunk_words).strip()

        if chunk and chunk not in seen:
            chunks.append(chunk)
            seen.add(chunk)

    return chunks


def build_chunk_df(movie_subset):
    rows = []

    for _, row in tqdm(movie_subset.iterrows(), total=len(movie_subset), desc="Building chunks"):
        chunks = screenplay_to_chunks(
            row["screenplay_text"],
            max_chunks=MAX_MOVIE_CHUNKS,
            words_per_chunk=WORDS_PER_CHUNK,
        )

        for chunk_idx, chunk in enumerate(chunks):
            rows.append({
                "movie_id": int(row["movie_id"]),
                "title": str(row[title_col]),
                "rating": float(row[rating_col]),
                "label": int(row["label"]),
                "chunk_idx": int(chunk_idx),
                "chunk_text": chunk,
            })

    return pd.DataFrame(rows)


# Movie-level split. No chunks from the same movie can appear in train and test.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_val_idx, test_idx = next(splitter.split(movie_df, movie_df["label"], groups=movie_df["norm_title"]))

train_val_movies = movie_df.iloc[train_val_idx].reset_index(drop=True)
test_movies = movie_df.iloc[test_idx].reset_index(drop=True)

val_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + 1)
train_idx, val_idx = next(val_splitter.split(
    train_val_movies,
    train_val_movies["label"],
    groups=train_val_movies["norm_title"],
))

train_movies = train_val_movies.iloc[train_idx].reset_index(drop=True)
val_movies = train_val_movies.iloc[val_idx].reset_index(drop=True)

print("Movie split sizes:")
print("Train movies:", len(train_movies))
print("Val movies:", len(val_movies))
print("Test movies:", len(test_movies))

train_chunks = build_chunk_df(train_movies)
val_chunks = build_chunk_df(val_movies)
test_chunks = build_chunk_df(test_movies)

print("Chunk split sizes:")
print("Train chunks:", len(train_chunks))
print("Val chunks:", len(val_chunks))
print("Test chunks:", len(test_chunks))

display(train_chunks.head())


Movie split sizes:
Train movies: 662
Val movies: 165
Test movies: 209


Building chunks:   0%|          | 0/662 [00:00<?, ?it/s]

Building chunks:   0%|          | 0/165 [00:00<?, ?it/s]

Building chunks:   0%|          | 0/209 [00:00<?, ?it/s]

Chunk split sizes:
Train chunks: 21184
Val chunks: 5280
Test chunks: 6688


,movie_id,title,rating,label,chunk_idx,chunk_text
0,0,10 Things I Hate About You,7.4,1,0,TEN THINGS I HATE ABOUT YOU written by Karen M...
1,0,10 Things I Hate About You,7.4,1,1,How many people were in your old school? CAMER...
2,0,10 Things I Hate About You,7.4,1,2,"worthy of our time She looks up at Ms. Blaise,..."
3,0,10 Things I Hate About You,7.4,1,3,Starving yourself is a very slow way to die. M...
4,0,10 Things I Hate About You,7.4,1,4,brakes to keep from hitting him KAT (yelling) ...


In [8]:
# ============================================================
# 8) DATASET + COLLATE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ScreenplayChunkDataset(Dataset):
    def __init__(self, chunk_df):
        self.texts = chunk_df["chunk_text"].astype(str).tolist()
        self.labels = chunk_df["label"].astype(int).tolist()
        self.movie_ids = chunk_df["movie_id"].astype(int).tolist()
        self.titles = chunk_df["title"].astype(str).tolist()
        self.ratings = chunk_df["rating"].astype(float).tolist()
        self.chunk_idxs = chunk_df["chunk_idx"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx],
            "movie_id": self.movie_ids[idx],
            "title": self.titles[idx],
            "rating": self.ratings[idx],
            "chunk_idx": self.chunk_idxs[idx],
        }


def collate_batch(batch):
    texts = [item["text"] for item in batch]

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_TOKEN_LENGTH,
        return_tensors="pt",
    )

    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
        "movie_ids": [item["movie_id"] for item in batch],
        "titles": [item["title"] for item in batch],
        "ratings": [item["rating"] for item in batch],
        "chunk_idxs": [item["chunk_idx"] for item in batch],
        "texts": texts,
    }


train_loader = DataLoader(
    ScreenplayChunkDataset(train_chunks),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    ScreenplayChunkDataset(val_chunks),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    ScreenplayChunkDataset(test_chunks),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# ============================================================
# 9) FINE-TUNED TRANSFORMER CLASSIFIER
# ============================================================

class TransformerMovieClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.25, unfreeze_last_n_layers=2):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        # Freeze everything first.
        for param in self.encoder.parameters():
            param.requires_grad = False

        # Unfreeze last layers for real fine-tuning without making laptop training insane.
        # DistilBERT: encoder.transformer.layer
        if hasattr(self.encoder, "transformer") and hasattr(self.encoder.transformer, "layer"):
            layers = self.encoder.transformer.layer
            for layer in layers[-unfreeze_last_n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True

        # BERT/RoBERTa style fallback: encoder.encoder.layer
        elif hasattr(self.encoder, "encoder") and hasattr(self.encoder.encoder, "layer"):
            layers = self.encoder.encoder.layer
            for layer in layers[-unfreeze_last_n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True

        # Always train the classifier head.
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 2),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # CLS token representation.
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled)

        return logits


model = TransformerMovieClassifier(
    MODEL_NAME,
    dropout=DROPOUT,
    unfreeze_last_n_layers=UNFREEZE_LAST_N_LAYERS,
).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,}")
print(f"Total params:     {total_params:,}")
print(f"Trainable %:      {100 * trainable_params / total_params:.2f}%")


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Trainable params: 14,471,810
Total params:     66,658,946
Trainable %:      21.71%


In [10]:
# ============================================================
# 10) TRAINING + MOVIE-LEVEL EVALUATION HELPERS
# ============================================================

train_label_counts = train_movies["label"].value_counts().sort_index()
class_counts = np.array([
    train_label_counts.get(0, 1),
    train_label_counts.get(1, 1),
], dtype=np.float32)

# Class weights stop the model from lazily predicting the majority class.
class_weights = class_counts.sum() / (2.0 * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = EPOCHS * len(train_loader)
warmup_steps = max(1, int(0.10 * total_steps))

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


def run_epoch_train(model, loader):
    model.train()

    total_loss = 0.0
    n_examples = 0

    for batch in tqdm(loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        n_examples += batch_size

    return total_loss / max(1, n_examples)


@torch.no_grad()
def predict_chunks(model, loader):
    model.eval()

    rows = []

    for batch in tqdm(loader, desc="Predicting chunks", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

        for i, prob in enumerate(probs):
            rows.append({
                "movie_id": batch["movie_ids"][i],
                "title": batch["titles"][i],
                "rating": batch["ratings"][i],
                "chunk_idx": batch["chunk_idxs"][i],
                "actual_label": int(batch["labels"][i].item()),
                "success_probability": float(prob),
                "chunk_text": batch["texts"][i],
            })

    return pd.DataFrame(rows)


def aggregate_movie_predictions(chunk_pred_df):
    grouped = chunk_pred_df.groupby("movie_id")

    rows = []

    for movie_id, group in grouped:
        probs = group["success_probability"].to_numpy()

        rows.append({
            "movie_id": movie_id,
            "title": group["title"].iloc[0],
            "rating": group["rating"].iloc[0],
            "actual_label": int(group["actual_label"].iloc[0]),
            "prob_mean": float(np.mean(probs)),
            "prob_median": float(np.median(probs)),
            "prob_max": float(np.max(probs)),
            "prob_min": float(np.min(probs)),
            "prob_std": float(np.std(probs)),
            "num_chunks": int(len(group)),
        })

    return pd.DataFrame(rows)


def choose_best_threshold(movie_pred_df, metric="f1"):
    y_true = movie_pred_df["actual_label"].to_numpy()
    probs = movie_pred_df["prob_mean"].to_numpy()

    best_threshold = 0.5
    best_score = -1.0

    for threshold in np.linspace(0.10, 0.90, 81):
        preds = (probs >= threshold).astype(int)

        if metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, preds)
        else:
            score = f1_score(y_true, preds, zero_division=0)

        if score > best_score:
            best_score = score
            best_threshold = threshold

    return float(best_threshold), float(best_score)


def evaluate_movie_predictions(movie_pred_df, threshold=0.5, name="Split"):
    y_true = movie_pred_df["actual_label"].to_numpy()
    probs = movie_pred_df["prob_mean"].to_numpy()
    preds = (probs >= threshold).astype(int)

    metrics = {
        "split": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, preds),
        "balanced_accuracy": balanced_accuracy_score(y_true, preds),
        "f1": f1_score(y_true, preds, zero_division=0),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, probs)
    else:
        metrics["roc_auc"] = np.nan

    metrics["prob_min"] = float(np.min(probs))
    metrics["prob_max"] = float(np.max(probs))
    metrics["prob_mean"] = float(np.mean(probs))

    return metrics, preds


def print_confusion(movie_pred_df, threshold):
    y_true = movie_pred_df["actual_label"].to_numpy()
    probs = movie_pred_df["prob_mean"].to_numpy()
    preds = (probs >= threshold).astype(int)

    cm = confusion_matrix(y_true, preds, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual < {RATING_THRESHOLD}", f"Actual >= {RATING_THRESHOLD}"],
        columns=[f"Pred < {RATING_THRESHOLD}", f"Pred >= {RATING_THRESHOLD}"],
    )

    display(cm_df)
    print(classification_report(
        y_true,
        preds,
        target_names=[f"< {RATING_THRESHOLD}", f">= {RATING_THRESHOLD}"],
        zero_division=0,
    ))


Class counts: [306. 356.]
Class weights: [1.0816994 0.9297753]


In [11]:
# ============================================================
# 11) TRAIN MODEL
# ============================================================

best_val_f1 = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch_train(model, train_loader)

    val_chunk_preds = predict_chunks(model, val_loader)
    val_movie_preds = aggregate_movie_predictions(val_chunk_preds)

    # Choose threshold using validation only.
    val_threshold, val_best_f1 = choose_best_threshold(val_movie_preds, metric="f1")
    val_metrics, _ = evaluate_movie_predictions(val_movie_preds, threshold=val_threshold, name="Val")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    }
    history.append(row)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val threshold: {val_threshold:.2f}")
    print(f"Val F1: {val_metrics['f1']:.4f}")
    print(f"Val balanced acc: {val_metrics['balanced_accuracy']:.4f}")
    print(f"Val ROC-AUC: {val_metrics['roc_auc']:.4f}")
    print(f"Val prob range: {val_metrics['prob_min']:.3f} to {val_metrics['prob_max']:.3f}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {
            "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "threshold": val_threshold,
            "epoch": epoch,
            "val_metrics": val_metrics,
        }

history_df = pd.DataFrame(history)
display(history_df)

if best_state is None:
    raise RuntimeError("Training failed to produce a best model state.")

model.load_state_dict(best_state["model_state_dict"])
best_threshold = best_state["threshold"]

print("Loaded best model from epoch:", best_state["epoch"])
print("Best validation threshold:", best_threshold)


Training:   0%|          | 0/2648 [00:00<?, ?it/s]

Predicting chunks:   0%|          | 0/660 [00:00<?, ?it/s]


Epoch 1/3
Train loss: 0.6188
Val threshold: 0.31
Val F1: 0.6996
Val balanced acc: 0.5284
Val ROC-AUC: 0.6198
Val prob range: 0.216 to 0.949


Training:   0%|          | 0/2648 [00:00<?, ?it/s]

Predicting chunks:   0%|          | 0/660 [00:00<?, ?it/s]

...

Training:   0%|          | 0/2648 [00:00<?, ?it/s]

Predicting chunks:   0%|          | 0/660 [00:00<?, ?it/s]


Epoch 3/3
Train loss: 0.2578
Val threshold: 0.10
Val F1: 0.6885
Val balanced acc: 0.5097
Val ROC-AUC: 0.6169
Val prob range: 0.026 to 0.997


,epoch,train_loss,split,threshold,accuracy,balanced_accuracy,f1,precision,recall,roc_auc,prob_min,prob_max,prob_mean
0,1,0.618820,Val,0.31,0.557576,0.528409,0.699588,0.548387,0.965909,0.619835,0.216221,0.949027,0.621652
1,2,0.387451,Val,0.12,0.551515,0.519481,0.704000,0.543210,1.000000,0.615407,0.080484,0.991328,0.589073
2,3,0.257834,Val,0.10,0.539394,0.509740,0.688525,0.538462,0.954545,0.616883,0.026410,0.996573,0.524275


Loaded best model from epoch: 2
Best validation threshold: 0.12000000000000001


In [12]:
# ============================================================
# 12) FINAL TEST EVALUATION
# ============================================================

test_chunk_preds = predict_chunks(model, test_loader)
test_movie_preds = aggregate_movie_predictions(test_chunk_preds)

test_metrics, test_binary_preds = evaluate_movie_predictions(
    test_movie_preds,
    threshold=best_threshold,
    name="Test",
)

print("=" * 80)
print("FINAL MOVIE-LEVEL TEST RESULTS")
print("=" * 80)
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\nConfusion matrix:")
print_confusion(test_movie_preds, best_threshold)

# Add final prediction column.
test_movie_preds["predicted_label"] = test_binary_preds
test_movie_preds["predicted_class"] = np.where(
    test_movie_preds["predicted_label"] == 1,
    f">= {RATING_THRESHOLD}",
    f"< {RATING_THRESHOLD}",
)

display(test_movie_preds.sort_values("prob_mean", ascending=False).head(20))
display(test_movie_preds.sort_values("prob_mean", ascending=True).head(20))


Predicting chunks:   0%|          | 0/836 [00:00<?, ?it/s]

FINAL MOVIE-LEVEL TEST RESULTS
split: Test
threshold: 0.1200
accuracy: 0.4880
balanced_accuracy: 0.4903
f1: 0.6537
precision: 0.4927
recall: 0.9712
roc_auc: 0.5687
prob_min: 0.0656
prob_max: 0.9945
prob_mean: 0.5967

Confusion matrix:


,Pred < 7.0,Pred >= 7.0
Actual < 7.0,1,104
Actual >= 7.0,3,101


              precision    recall  f1-score   support

       < 7.0       0.25      0.01      0.02       105
      >= 7.0       0.49      0.97      0.65       104

    accuracy                           0.49       209
   macro avg       0.37      0.49      0.34       209
weighted avg       0.37      0.49      0.33       209



,movie_id,title,rating,actual_label,prob_mean,prob_median,prob_max,prob_min,prob_std,num_chunks,predicted_label,predicted_class
195,977,Valkyrie,7.1,1,0.994523,0.997081,0.998559,0.955170,0.007892,32,1,>= 7.0
181,914,Taking Sides,7.1,1,0.988029,0.992515,0.997596,0.923847,0.014203,32,1,>= 7.0
128,624,Mary Poppins,7.8,1,0.973758,0.994895,0.998042,0.438981,0.096919,32,1,>= 7.0
192,958,Tristan & Isolde,7.3,1,0.971556,0.979596,0.997137,0.899125,0.023522,32,1,>= 7.0
19,89,As Good As It Gets,7.7,1,0.970169,0.993532,0.997575,0.557985,0.080489,32,1,>= 7.0
16,73,Anonymous,6.8,0,0.966389,0.985153,0.996112,0.850043,0.040191,32,1,>= 7.0
80,414,The Godfather Part II,9.0,1,0.965262,0.987666,0.996943,0.776326,0.049014,32,1,>= 7.0
106,538,Kafka,6.8,0,0.962826,0.987814,0.997321,0.666185,0.075665,32,1,>= 7.0
201,1004,The White Ribbon,7.8,1,0.960185,0.984914,0.997801,0.699810,0.069391,32,1,>= 7.0
182,916,Tall in the Saddle,6.9,0,0.957078,0.976281,0.998013,0.600930,0.069928,32,1,>= 7.0


,movie_id,title,rating,actual_label,prob_mean,prob_median,prob_max,prob_min,prob_std,num_chunks,predicted_label,predicted_class
23,100,The Avengers,8.0,1,0.065634,0.020146,0.512207,0.005908,0.113798,32,0,< 7.0
22,99,The Avengers,8.0,1,0.065634,0.020146,0.512207,0.005908,0.113798,32,0,< 7.0
47,262,The Crow,7.5,1,0.112322,0.032002,0.744093,0.007787,0.192963,32,0,< 7.0
149,751,Priest,5.7,0,0.119028,0.084677,0.459367,0.007793,0.108466,32,0,< 7.0
93,472,Horrible Bosses,6.9,0,0.124387,0.064425,0.814943,0.007955,0.156926,32,1,>= 7.0
176,888,Stepmom,6.8,0,0.140801,0.067437,0.723877,0.007689,0.173569,32,1,>= 7.0
118,595,The Losers,6.2,0,0.143343,0.044878,0.629807,0.010242,0.190037,32,1,>= 7.0
58,311,Drop Dead Gorgeous,6.7,0,0.143712,0.084049,0.706581,0.010995,0.152406,32,1,>= 7.0
161,828,Sex and the City,7.4,1,0.156586,0.070160,0.800127,0.009913,0.182821,32,1,>= 7.0
78,393,Gamer,5.7,0,0.157365,0.051991,0.817209,0.007510,0.211555,32,1,>= 7.0


In [13]:
# ============================================================
# 13) BASELINE COMPARISON
# ============================================================

# Majority-class baseline on test movies.
majority_label = int(train_movies["label"].mode().iloc[0])
baseline_preds = np.full(len(test_movie_preds), majority_label)

y_test = test_movie_preds["actual_label"].to_numpy()

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_preds),
    "balanced_accuracy": balanced_accuracy_score(y_test, baseline_preds),
    "f1": f1_score(y_test, baseline_preds, zero_division=0),
    "precision": precision_score(y_test, baseline_preds, zero_division=0),
    "recall": recall_score(y_test, baseline_preds, zero_division=0),
}

comparison_df = pd.DataFrame([
    {"model": "Majority baseline", **baseline_metrics},
    {"model": "Fine-tuned transformer", **{k: v for k, v in test_metrics.items() if k in baseline_metrics}},
])

display(comparison_df)


,model,accuracy,balanced_accuracy,f1,precision,recall
0,Majority baseline,0.497608,0.500000,0.664537,0.497608,1.000000
1,Fine-tuned transformer,0.488038,0.490339,0.653722,0.492683,0.971154


In [14]:
# ============================================================
# 14) THEME / CHUNK ANALYSIS
# ============================================================

# Goal:
# Find terms that are more common in chunks the model thinks are "successful"
# versus chunks it thinks are "unsuccessful".
#
# This is not a perfect "theme detector", but it gives interpretable clues.

analysis_chunks = test_chunk_preds.copy()

high_chunks = analysis_chunks[analysis_chunks["success_probability"] >= 0.75]
low_chunks = analysis_chunks[analysis_chunks["success_probability"] <= 0.25]

print("High-confidence successful chunks:", len(high_chunks))
print("High-confidence unsuccessful chunks:", len(low_chunks))

if len(high_chunks) >= 5 and len(low_chunks) >= 5:
    theme_df = pd.concat([
        high_chunks.assign(pred_group="high_success"),
        low_chunks.assign(pred_group="low_success"),
    ], ignore_index=True)

    vectorizer = TfidfVectorizer(
        max_features=3000,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
    )

    X_tfidf = vectorizer.fit_transform(theme_df["chunk_text"])
    feature_names = np.array(vectorizer.get_feature_names_out())

    high_mask = theme_df["pred_group"].to_numpy() == "high_success"
    low_mask = theme_df["pred_group"].to_numpy() == "low_success"

    high_mean = np.asarray(X_tfidf[high_mask].mean(axis=0)).ravel()
    low_mean = np.asarray(X_tfidf[low_mask].mean(axis=0)).ravel()

    diff = high_mean - low_mean

    top_high_idx = np.argsort(diff)[-30:][::-1]
    top_low_idx = np.argsort(diff)[:30]

    theme_terms_df = pd.DataFrame({
        "more_successful_terms": feature_names[top_high_idx],
        "success_term_score": diff[top_high_idx],
        "more_unsuccessful_terms": feature_names[top_low_idx],
        "unsuccess_term_score": -diff[top_low_idx],
    })

    display(theme_terms_df)

else:
    print("Not enough high/low confidence chunks for TF-IDF theme comparison.")
    print("Try using thresholds 0.65 / 0.35 instead.")

# Show examples the model is very confident about.
print("\nMost successful-looking chunks:")
display(
    analysis_chunks.sort_values("success_probability", ascending=False)
    [["title", "rating", "actual_label", "success_probability", "chunk_idx", "chunk_text"]]
    .head(8)
)

print("\nMost unsuccessful-looking chunks:")
display(
    analysis_chunks.sort_values("success_probability", ascending=True)
    [["title", "rating", "actual_label", "success_probability", "chunk_idx", "chunk_text"]]
    .head(8)
)


High-confidence successful chunks: 3111
High-confidence unsuccessful chunks: 1777


,more_successful_terms,success_term_score,more_unsuccessful_terms,unsuccess_term_score
0,sam,0.011890,emma,0.016227
1,jacob,0.010199,rachel,0.012802
2,david,0.007726,grace,0.012763
3,mr,0.007612,steed,0.010513
4,richard,0.007289,eric,0.009510
5,scott,0.007257,ace,0.009213
6,kafka,0.005785,continued,0.009148
7,marty,0.005767,nick,0.008454
8,joel,0.005308,kathy,0.008395
9,guido,0.005146,lilly,0.007954



Most successful-looking chunks:


,title,rating,actual_label,success_probability,chunk_idx,chunk_text
6269,Valkyrie,7.1,1,0.998559,29,"pompous, know-it-all General. WITZLEBEN The mi..."
6256,Valkyrie,7.1,1,0.998304,16,- BUNKER - HALLWAY - DAY 63 CLOSE ON A CIGARET...
6267,Valkyrie,7.1,1,0.998214,27,think? REMER Of that I am certain. What I can'...
6259,Valkyrie,7.1,1,0.998123,19,"But that's not what he- OLBRICHT Noted, Colone..."
4932,Reservoir Dogs,8.3,1,0.998104,4,"to help you. We're just gonna sit here, and wa..."
6244,Valkyrie,7.1,1,0.998069,4,can't honestly believe you'll make a differenc...
6251,Valkyrie,7.1,1,0.998065,11,privacy. DRUNKEN OFFICER Are you two finished ...
3425,Kate & Leopold,6.4,0,0.998060,1,waffle: LEOPOLD You dance like a herd of cattl...



Most unsuccessful-looking chunks:


,title,rating,actual_label,success_probability,chunk_idx,chunk_text
260,Alone in the Dark,2.4,0,0.002460,4,approaches the crate. DELIVERY GUY Says here i...
5896,The Rage: Carrie 2,4.8,0,0.004410,8,on the counter as the metronome of the sink fa...
2835,He's Just Not That Into You,6.4,0,0.005087,19,Just call. Mary cradles the receiver with her ...
6425,The Whistleblower,7.1,1,0.005342,25,to be afraid of Ivan Bladzic. He cannot hurt y...
2356,Final Destination,6.7,0,0.005508,20,into the hot mug. CAMERA PUSHES INTO THE MUG a...
6511,Wild Hogs,5.9,0,0.005575,15,(CONT'D) (TO GUYS) She's perfect. DOUG You lik...
5955,There's Something About Mary,7.1,1,0.005710,3,"NOW, he hastily zips up his fly and TED YEEEOO..."
1932,Dune,6.3,0,0.005826,12,"many things. (out loud, suddenly) Is there a r..."


In [15]:
# ============================================================
# 15) SAVE OUTPUTS
# ============================================================

test_movie_preds.to_csv(PREDICTIONS_OUT_PATH, index=False)
test_chunk_preds.to_csv(CHUNK_PREDICTIONS_OUT_PATH, index=False)

torch.save({
    "model_state_dict": model.state_dict(),
    "model_name": MODEL_NAME,
    "rating_threshold": RATING_THRESHOLD,
    "best_threshold": best_threshold,
    "config": {
        "max_movie_chunks": MAX_MOVIE_CHUNKS,
        "words_per_chunk": WORDS_PER_CHUNK,
        "max_token_length": MAX_TOKEN_LENGTH,
        "unfreeze_last_n_layers": UNFREEZE_LAST_N_LAYERS,
    },
    "test_metrics": test_metrics,
}, MODEL_OUT_PATH)

print("Saved movie predictions to:", PREDICTIONS_OUT_PATH)
print("Saved chunk predictions to:", CHUNK_PREDICTIONS_OUT_PATH)
print("Saved model to:", MODEL_OUT_PATH)


Saved movie predictions to: bert_success_classifier_movie_predictions.csv
Saved chunk predictions to: bert_success_classifier_chunk_predictions.csv
Saved model to: bert_success_classifier_model.pt


## How to interpret the result

This model should not be judged by exact predicted rating. It outputs a **probability of success**.

Important checks:

1. **Does it beat the majority baseline?**
2. **Is balanced accuracy above 0.50?**
3. **Is ROC-AUC above 0.50?**
4. **Is the probability range wide?**
   - Bad: probabilities all stuck around `0.45–0.55`
   - Better: probabilities spread like `0.15–0.90`

If it still barely beats baseline, the honest project conclusion is:

> Screenplay text alone contains weak signal for IMDb success, but exact success is heavily influenced by non-screenplay factors.
